# 05.5: Final Experiment — GCR Baseline vs DCA-Trie v1 vs DCA-Trie v2

**Purpose:** Run all three conditions on both WebQSP and CWQ (50 samples each)
and produce comparison results for the thesis writeup.

**Conditions:**
1. **GCR Baseline** — unfiltered DFS paths, standard constrained decoding
2. **DCA-Trie v1 (Static)** — TypeOracle pre-filters all paths, then builds trie
3. **DCA-Trie v2 (Dynamic)** — iterative hop-by-hop trie expansion with symbolic gates

**Datasets:** RoG-webqsp, RoG-cwq
**Samples:** 50 per dataset
**Model:** rmanluo/GCR-Meta-Llama-3.1-8B-Instruct

In [ ]:
# @title 1. Setup & Environment
import os
import sys
import time
import json
import warnings
import argparse
import copy

warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive', force_remount=False)
    DRIVE_BASE = "/content/drive/MyDrive/GCR_FinalExperiment"
    os.makedirs(DRIVE_BASE, exist_ok=True)
except ImportError:
    IN_COLAB = False
    DRIVE_BASE = None
    print("Not on Colab — skipping Drive mount.")

import torch

GPU_ARCH_MAP = {
    "A100": "80", "A100-SXM": "80", "A100-PCIE": "80",
    "L4": "89", "T4": "75", "V100": "70",
    "H100": "90", "H200": "90",
    "RTX 4090": "89", "RTX 3090": "86",
}

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_arch = None
    for key, arch in GPU_ARCH_MAP.items():
        if key in gpu_name:
            gpu_arch = arch
            break
    has_a100 = "A100" in gpu_name
    print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB  |  SM: {gpu_arch}")
else:
    gpu_name = "None"
    gpu_mem = 0
    gpu_arch = None
    has_a100 = False
    print("WARNING: No GPU detected.")

REPO_DIR = "/content/graph-constrained-reasoning" if IN_COLAB else os.getcwd()
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Adjanour/graph-constrained-reasoning-dca {REPO_DIR}
if IN_COLAB:
    os.chdir(REPO_DIR)
sys.path.insert(0, '.')

DEPS = [
    "transformers==4.44.2", "accelerate>=0.30.1", "datasets>=2.19.2",
    "marisa-trie>=1.2.0", "networkx>=3.0", "scikit-learn>=1.5.0",
    "tiktoken>=0.7.0", "sentencepiece>=0.2.0", "protobuf>=5.27.1",
]
print("Installing dependencies...")
!pip install -q {' '.join(DEPS)}

# flash-attn: pre-built wheels only
flash_attn_installed = False
if torch.cuda.is_available():
    py_ver = f"{sys.version_info.major}{sys.version_info.minor}"
    cuda_ver = torch.version.cuda.replace(".", "")
    torch_ver = torch.__version__.split("+")[0]
    torch_mm = ".".join(torch_ver.split(".")[:2])
    for abi in ["FALSE", "TRUE"]:
        for tt in [torch_mm, torch_ver]:
            wheel_url = (
                f"https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/"
                f"flash_attn-2.8.3+cu{cuda_ver}torch{tt}cxx11abi{abi}"
                f"-cp{py_ver}-cp{py_ver}-linux_x86_64.whl"
            )
            ret = os.system(f"pip install -q '{wheel_url}' 2>/dev/null")
            if ret == 0:
                flash_attn_installed = True
                break
        if flash_attn_installed:
            break
    if flash_attn_installed:
        print("flash-attn installed.")
    else:
        print("flash-attn unavailable — using sdpa.")

from tqdm import tqdm
from datasets import load_dataset
from src.llms import get_registed_model
from src.qa_prompt_builder import PathGenerationWithAnswerPromptBuilder
from src.trie import MarisaTrie
from src.graph_constrained_decoding import GraphConstrainedDecoding
from src.utils.qa_utils import eval_path_result_w_ans, normalize, extract_topk_prediction
from approach3_symbolic.type_oracle import TypeOracle
import src.utils as utils
import networkx as nx

print("All imports verified.")

---
## 2. Configuration

In [ ]:
# @title 2. Configuration

MODEL_PATH = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"
DATA_PATH = "rmanluo"
DATASETS = ["RoG-webqsp", "RoG-cwq"]
SPLIT = "test"
INDEX_LEN = 2
K = 10
GEN_MODE = "group-beam"
PROMPT_MODE = "zero-shot"
MAX_NEW_TOKENS = 256
MAX_SAMPLES = 50
FORCE_RERUN = False

ATTN_IMPL = "flash_attention_2" if (has_a100 and flash_attn_installed) else "sdpa"
TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")

if IN_COLAB:
    OUTPUT_BASE = os.path.join(DRIVE_BASE, TIMESTAMP)
else:
    OUTPUT_BASE = os.path.join("results", "final_experiment", TIMESTAMP)
os.makedirs(OUTPUT_BASE, exist_ok=True)

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Model:       {MODEL_PATH}")
print(f"Datasets:    {DATASETS}")
print(f"Split:       {SPLIT}")
print(f"Samples:     {MAX_SAMPLES} per dataset")
print(f"Beam width:  {K}")
print(f"Index len:   {INDEX_LEN} hops")
print(f"Generation:  {GEN_MODE}")
print(f"Attention:   {ATTN_IMPL}")
print(f"GPU:         {gpu_name}")
print(f"Output:      {OUTPUT_BASE}")
print("=" * 60)

---
## 3. Load LLM

In [ ]:
# @title 3. Load Model

LLM = get_registed_model(MODEL_PATH)
parser = argparse.ArgumentParser()
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.model_name = MODEL_PATH
args.k = K
args.generation_mode = GEN_MODE
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = MAX_NEW_TOKENS
args.maximun_token = 4096

print(f"Loading {MODEL_PATH}...")
t0 = time.time()
model = LLM(args)
model.prepare_for_inference()
model.generation_cfg.temperature = None
model.generation_cfg.top_p = None
model.generation_cfg.top_k = None
model.model.generation_config.temperature = None
model.model.generation_config.top_p = None
model.model.generation_config.top_k = None

load_time = time.time() - t0
n_params = sum(p.numel() for p in model.model.parameters()) / 1e6
vram = torch.cuda.memory_allocated(0) / 1e9 if torch.cuda.is_available() else 0
print(f"Loaded in {load_time:.1f}s  |  {n_params:.1f}M params  |  {vram:.1f} GB VRAM")

---
## 4. Helper Functions

In [ ]:
# @title 4. Helpers

PATH_START = "<PATH>"
PATH_END = "</PATH>"


def build_filtered_trie(tokenizer, question_dict, index_len, oracle):
    """Build a MarisaTrie from TypeOracle-filtered paths (v1 static)."""
    g = utils.build_graph(question_dict["graph"], undirected=False)
    entities = question_dict.get("q_entity", [])
    if not entities:
        return None, [], []

    all_paths = utils.dfs(g, entities, index_len)
    ans_types = oracle.infer_answer_types(question_dict["question"])

    filtered = []
    for p in all_paths:
        admit = True
        for _, rel, tail in p:
            if not oracle.range_gate(rel, tail):
                admit = False
                break
        if admit:
            terminal = p[-1][2]
            if not oracle.type_gate(terminal, ans_types, len(p), index_len):
                admit = False
        if admit:
            filtered.append(p)

    filtered_str = [utils.path_to_string(p) for p in filtered]
    if not filtered_str:
        return None, all_paths, filtered

    wrapped = [f"{PATH_START}{s}{PATH_END}" for s in filtered_str]
    tokenized = tokenizer(wrapped, padding=False, add_special_tokens=False).input_ids
    trie = MarisaTrie(tokenized, max_token_id=len(tokenizer) + 1)
    return trie, all_paths, filtered


def build_unfiltered_trie(tokenizer, question_dict, index_len):
    """Build a MarisaTrie from all DFS paths (GCR baseline)."""
    g = utils.build_graph(question_dict["graph"], undirected=False)
    entities = question_dict.get("q_entity", [])
    if not entities:
        return None, []

    all_paths = utils.dfs(g, entities, index_len)
    all_str = [utils.path_to_string(p) for p in all_paths]
    if not all_str:
        return None, all_paths

    wrapped = [f"{PATH_START}{s}{PATH_END}" for s in all_str]
    tokenized = tokenizer(wrapped, padding=False, add_special_tokens=False).input_ids
    trie = MarisaTrie(tokenized, max_token_id=len(tokenizer) + 1)
    return trie, all_paths


def run_constrained_decoding(model, input_builder, data, trie):
    """Run graph-constrained decoding for a single question."""
    input_query, ground_paths, _ = input_builder.process_input(data, return_tire=False)
    start_token_ids = model.tokenizer.convert_tokens_to_ids(input_builder.PATH_START_TOKEN)
    end_token_ids = model.tokenizer.convert_tokens_to_ids(input_builder.PATH_END_TOKEN)
    llm_input = model.prepare_model_prompt(input_query)
    prediction = model.generate_sentence(
        llm_input, trie,
        start_token_ids=start_token_ids,
        end_token_ids=end_token_ids,
        enable_constrained_by_default=False,
    )
    return prediction, ground_paths


def build_path_trie_from_strings(tokenizer, path_strings):
    """Build a MarisaTrie from raw path strings (for v2 iterative expansion)."""
    if not path_strings:
        return None
    wrapped = [f"{PATH_START}{s}{PATH_END}" for s in path_strings]
    tokenized = tokenizer(wrapped, padding=False, add_special_tokens=False).input_ids
    tokenized = [ids + [tokenizer.eos_token_id] for ids in tokenized]
    return MarisaTrie(tokenized, max_token_id=len(tokenizer) + 1)


def extract_entity_from_output(output_text):
    """Extract the last committed entity from generated text."""
    cleaned = output_text.replace(PATH_END, "").strip()
    parts = cleaned.split(" -> ")
    return parts[-1].strip() if parts else None


def dca_v2_generate(question, start_entities, nx_graph, llm_model,
                    tokenizer, oracle, input_builder, max_hops=2):
    """
    DCA-Trie v2: iterative hop-by-hop trie expansion (Algorithm 2).
    Start with first-hop gated neighbours, expand at each entity commit.
    """
    answer_types = oracle.infer_answer_types(question)
    start_id = tokenizer.convert_tokens_to_ids(PATH_START)
    end_id = tokenizer.convert_tokens_to_ids(PATH_END)

    # Initial trie: first-hop gated neighbours
    first_hop_paths = []
    for entity in start_entities:
        if entity not in nx_graph:
            continue
        for neighbor in nx_graph.neighbors(entity):
            rel = nx_graph[entity][neighbor]["relation"]
            if not oracle.range_gate(rel, neighbor):
                continue
            first_hop_paths.append(f"{entity} -> {rel} -> {neighbor}")

    if not first_hop_paths:
        return None, []

    current_trie = build_path_trie_from_strings(tokenizer, first_hop_paths)
    if current_trie is None:
        return None, []

    # Build prompt
    input_query, ground_paths, _ = input_builder.process_input(question_dict, return_tire=False) if False else ("", [], [])
    prompt = (
        f"Reasoning path is a sequence of triples in the KG that connects the topic entities "
        f"to answer entities. Given the question, generate reasoning paths starting from "
        f"the topic entities to answer the question.\n\n"
        f"# Question:\n{question}\n"
        f"# Topic entities:\n{', '.join(start_entities)}\n"
    )

    output_text = ""
    committed_entity = start_entities[0] if start_entities else None
    hop = 0

    for step in range(max_hops * 3):
        llm_input = llm_model.prepare_model_prompt(prompt)
        gcr = GraphConstrainedDecoding(
            tokenizer, current_trie, start_id, end_id,
            enable_constrained_by_default=False
        )
        inputs = tokenizer(llm_input, return_tensors="pt", add_special_tokens=False)
        input_ids = inputs.input_ids.to(llm_model.model.device)
        attn_mask = inputs.attention_mask.to(llm_model.model.device)

        res = llm_model.model.generate(
            input_ids=input_ids,
            attention_mask=attn_mask,
            generation_config=llm_model.generation_cfg,
            prefix_allowed_tokens_fn=gcr.allowed_tokens_fn,
            return_dict_in_generate=True,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=32,
        )

        output = tokenizer.decode(
            res.sequences[0][input_ids.shape[1]:], skip_special_tokens=True
        )
        output_text += output

        if PATH_END in output or tokenizer.eos_token in output:
            break

        new_entity = extract_entity_from_output(output)
        if new_entity is None or new_entity == committed_entity:
            continue
        committed_entity = new_entity
        hop += 1

        if hop >= max_hops:
            break

        # Expand trie with gated neighbours
        is_terminal = (hop + 1 >= max_hops)
        new_paths = []
        if committed_entity in nx_graph:
            for neighbor in nx_graph.neighbors(committed_entity):
                rel = nx_graph[committed_entity][neighbor]["relation"]
                if not oracle.range_gate(rel, neighbor):
                    continue
                if is_terminal and not oracle.type_gate(
                    neighbor, answer_types, hop + 1, max_hops
                ):
                    continue
                new_paths.append(f"{committed_entity} -> {rel} -> {neighbor}")

        if new_paths:
            expanded_trie = build_path_trie_from_strings(tokenizer, new_paths)
            if expanded_trie is not None:
                current_trie = expanded_trie
                prompt = prompt + f"\n{output.strip()}\n"
        else:
            break

    return output_text, ground_paths


print("Helpers defined.")

---
## 5. Run All Conditions

In [ ]:
# @title 5. Run Experiment

all_results = {}  # {(dataset, condition): {hits, n, time, ...}}

for ds_name in DATASETS:
    print(f"\n{'#' * 60}")
    print(f"  DATASET: {ds_name}")
    print(f"{'#' * 60}")

    dataset = load_dataset(f"{DATA_PATH}/{ds_name}", split=SPLIT)
    if MAX_SAMPLES and MAX_SAMPLES < len(dataset):
        dataset = dataset.select(range(MAX_SAMPLES))
    print(f"  Samples: {len(dataset)}")

    ds_dir = os.path.join(OUTPUT_BASE, ds_name)
    os.makedirs(ds_dir, exist_ok=True)

    input_builder = PathGenerationWithAnswerPromptBuilder(
        model.tokenizer, PROMPT_MODE, index_path_length=INDEX_LEN
    )

    # ── Condition 1: GCR Baseline ────────────────────────────────
    cond_name = "GCR_Baseline"
    pred_path = os.path.join(ds_dir, f"predictions_{cond_name}.jsonl")
    processed = set()
    if not FORCE_RERUN and os.path.exists(pred_path):
        with open(pred_path) as f:
            for line in f:
                try:
                    processed.add(json.loads(line)["id"])
                except: pass

    fout = open(pred_path, "a" if processed else "w")
    n_done = 0
    t0 = time.time()

    for d in dataset:
        if d["id"] in processed:
            continue
        trie, all_paths = build_unfiltered_trie(model.tokenizer, d, INDEX_LEN)
        if trie is None:
            result = {"id": d["id"], "question": d["question"],
                      "prediction": [], "ground_truth": d["answer"],
                      "n_paths_all": 0, "mode": cond_name}
            fout.write(json.dumps(result) + "\n")
            fout.flush()
            processed.add(d["id"])
            n_done += 1
            continue
        try:
            prediction, ground_paths = run_constrained_decoding(model, input_builder, d, trie)
        except Exception as e:
            prediction = None
        result = {"id": d["id"], "question": d["question"],
                  "prediction": prediction or [], "ground_truth": d["answer"],
                  "n_paths_all": len(all_paths), "mode": cond_name}
        fout.write(json.dumps(result) + "\n")
        fout.flush()
        processed.add(d["id"])
        n_done += 1
        if n_done % 10 == 0:
            print(f"    [{cond_name}] {n_done}/{len(dataset)} ({time.time()-t0:.0f}s)")

    fout.close()
    elapsed_gcr = time.time() - t0
    print(f"    {cond_name}: {n_done} done in {elapsed_gcr:.0f}s")

    # ── Condition 2: DCA-Trie v1 (Static) ───────────────────────
    cond_name = "DCA_v1_Static"
    pred_path = os.path.join(ds_dir, f"predictions_{cond_name}.jsonl")
    processed = set()
    if not FORCE_RERUN and os.path.exists(pred_path):
        with open(pred_path) as f:
            for line in f:
                try:
                    processed.add(json.loads(line)["id"])
                except: pass

    fout = open(pred_path, "a" if processed else "w")
    n_done = 0
    t0 = time.time()

    for d in dataset:
        if d["id"] in processed:
            continue
        oracle = TypeOracle.from_graph(d["graph"])
        trie, all_paths, filtered = build_filtered_trie(model.tokenizer, d, INDEX_LEN, oracle)
        if trie is None:
            result = {"id": d["id"], "question": d["question"],
                      "prediction": [], "ground_truth": d["answer"],
                      "n_paths_all": len(all_paths) if all_paths else 0,
                      "n_paths_filtered": 0, "mode": cond_name}
            fout.write(json.dumps(result) + "\n")
            fout.flush()
            processed.add(d["id"])
            n_done += 1
            continue
        try:
            prediction, ground_paths = run_constrained_decoding(model, input_builder, d, trie)
        except Exception as e:
            prediction = None
        result = {"id": d["id"], "question": d["question"],
                  "prediction": prediction or [], "ground_truth": d["answer"],
                  "n_paths_all": len(all_paths), "n_paths_filtered": len(filtered),
                  "mode": cond_name}
        fout.write(json.dumps(result) + "\n")
        fout.flush()
        processed.add(d["id"])
        n_done += 1
        if n_done % 10 == 0:
            print(f"    [{cond_name}] {n_done}/{len(dataset)} ({time.time()-t0:.0f}s)")

    fout.close()
    elapsed_v1 = time.time() - t0
    print(f"    {cond_name}: {n_done} done in {elapsed_v1:.0f}s")

    # ── Condition 3: DCA-Trie v2 (Dynamic) ──────────────────────
    cond_name = "DCA_v2_Dynamic"
    pred_path = os.path.join(ds_dir, f"predictions_{cond_name}.jsonl")
    processed = set()
    if not FORCE_RERUN and os.path.exists(pred_path):
        with open(pred_path) as f:
            for line in f:
                try:
                    processed.add(json.loads(line)["id"])
                except: pass

    fout = open(pred_path, "a" if processed else "w")
    n_done = 0
    n_dead_ends = 0
    t0 = time.time()

    for d in dataset:
        if d["id"] in processed:
            continue
        oracle = TypeOracle.from_graph(d["graph"])
        nx_graph = utils.build_graph(d["graph"], undirected=False)
        try:
            prediction, ground_paths = dca_v2_generate(
                question=d["question"],
                start_entities=d.get("q_entity", []),
                nx_graph=nx_graph,
                llm_model=model,
                tokenizer=model.tokenizer,
                oracle=oracle,
                input_builder=input_builder,
                max_hops=INDEX_LEN,
            )
            if prediction is None:
                n_dead_ends += 1
        except Exception as e:
            prediction = None

        result = {"id": d["id"], "question": d["question"],
                  "prediction": prediction or [], "ground_truth": d["answer"],
                  "mode": cond_name}
        fout.write(json.dumps(result) + "\n")
        fout.flush()
        processed.add(d["id"])
        n_done += 1
        if n_done % 10 == 0:
            print(f"    [{cond_name}] {n_done}/{len(dataset)} ({time.time()-t0:.0f}s)")

    fout.close()
    elapsed_v2 = time.time() - t0
    print(f"    {cond_name}: {n_done} done in {elapsed_v2:.0f}s ({n_dead_ends} dead ends)")

print(f"\n{'=' * 60}")
print("All conditions complete.")
print(f"{'=' * 60}")

---
## 6. Evaluate & Compare

In [ ]:
# @title 6. Evaluation

def load_preds(path):
    results = []
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                try:
                    results.append(json.loads(line))
                except: pass
    return results


def compute_hits(preds):
    hits = 0
    for p in preds:
        prediction = p.get("prediction", "")
        answer = list(set(p.get("ground_truth", [])))
        pred_str = " ".join(prediction) if isinstance(prediction, list) else prediction
        top_preds = extract_topk_prediction(pred_str, -1)
        pred_joined = " ".join(top_preds)
        for a in answer:
            if normalize(a) in normalize(pred_joined):
                hits += 1
                break
    return hits


print("=" * 70)
print(f"{'Dataset':<15} {'Condition':<20} {'N':<6} {'Hits@1':<8} {'Hit%':<8} {'Time(s)':<8}")
print("-" * 70)

summary = {}

for ds_name in DATASETS:
    ds_dir = os.path.join(OUTPUT_BASE, ds_name)
    for cond in ["GCR_Baseline", "DCA_v1_Static", "DCA_v2_Dynamic"]:
        pred_path = os.path.join(ds_dir, f"predictions_{cond}.jsonl")
        preds = load_preds(pred_path)
        n = len(preds)
        hits = compute_hits(preds)
        hit_pct = hits / max(1, n) * 100

        # Path reduction for v1
        if cond == "DCA_v1_Static" and n > 0:
            total_all = sum(p.get("n_paths_all", 0) for p in preds)
            total_filt = sum(p.get("n_paths_filtered", 0) for p in preds)
            reduction = (1 - total_filt / max(1, total_all)) * 100
            print(f"{ds_name:<15} {cond:<20} {n:<6} {hits:<8} {hit_pct:<8.1f} {'':>8} (paths: {total_filt}/{total_all}, -{reduction:.1f}%)")
        else:
            print(f"{ds_name:<15} {cond:<20} {n:<6} {hits:<8} {hit_pct:<8.1f}")

        summary[(ds_name, cond)] = {"n": n, "hits": hits, "hit_pct": hit_pct}

print("=" * 70)

---
## 7. Save Results

In [ ]:
# @title 7. Save Everything

config = {
    "model": MODEL_PATH, "datasets": DATASETS, "split": SPLIT,
    "max_samples": MAX_SAMPLES, "index_len": INDEX_LEN, "k": K,
    "gen_mode": GEN_MODE, "prompt_mode": PROMPT_MODE,
    "attn_impl": ATTN_IMPL, "gpu": gpu_name, "timestamp": TIMESTAMP,
}
with open(os.path.join(OUTPUT_BASE, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

with open(os.path.join(OUTPUT_BASE, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

# Markdown report
report_lines = [
    f"# Final Experiment Results\n",
    f"**Timestamp:** {TIMESTAMP}\n",
    f"**Model:** {MODEL_PATH}\n",
    f"**Samples:** {MAX_SAMPLES} per dataset\n\n",
    f"## Results\n\n",
    f"| Dataset | Condition | N | Hits@1 | Hit% |\n",
    f"|---------|-----------|---|--------|------|\n",
]
for (ds, cond), m in summary.items():
    report_lines.append(f"| {ds} | {cond} | {m['n']} | {m['hits']} | {m['hit_pct']:.1f}% |\n")

with open(os.path.join(OUTPUT_BASE, "report.md"), "w") as f:
    f.writelines(report_lines)

print(f"Results saved to {OUTPUT_BASE}")
print("\nFiles:")
for fname in sorted(os.listdir(OUTPUT_BASE)):
    fpath = os.path.join(OUTPUT_BASE, fname)
    if os.path.isfile(fpath):
        print(f"  {fname}")
    else:
        for sub in sorted(os.listdir(fpath)):
            print(f"  {fname}/{sub}")